# Импорт данных с экономией памяти

## Принципы импорта

Опираемся на текущую документацию и на фактические CSV.

Что делаем для экономии памяти:
- не читаем `Unnamed: 0`;
- пропускаем поля, которые в документации помечены как ненужные, а также тяжёлые детали вроде `results`;
- читаем числа с `thousands=","`, потому что ID в CSV записаны с разделителями тысяч;
- сразу переводим low-cardinality строки в `category`;
- downcast-им числовые столбцы до `Int8/Int16/Int32` и `Float32`, где это безопасно.


In [ ]:
from pathlib import Path

import pandas as pd

SRC_DIR = Path.cwd() / "src"


In [ ]:
# Базовые таблицы: пользователи, курсы и структура курса.
users_df = pd.read_csv(
    SRC_DIR / "users.csv",
    usecols=[
        "id",
        "last_explainer_seen_→_course",
        "created_at",
        "updated_at",
        "type",
        "sign_in_count",
        "grade_id",
        "subscribed",
        "is_teacher",
        "timezone",
        "grade_changed_at",
        "d_wk_school_id",
        "d_wk_municipal_id",
        "d_wk_region_id",
        "wk_gender",
    ],
    parse_dates=["created_at", "updated_at", "grade_changed_at"],
    thousands=",",
    low_memory=False,
)
users_df["id"] = users_df["id"].astype("Int32")
users_df["last_explainer_seen_→_course"] = users_df["last_explainer_seen_→_course"].astype("Int8")
users_df["sign_in_count"] = users_df["sign_in_count"].astype("Int32")
users_df["grade_id"] = users_df["grade_id"].astype("Int16")
users_df["d_wk_school_id"] = users_df["d_wk_school_id"].astype("Int32")
users_df["d_wk_municipal_id"] = users_df["d_wk_municipal_id"].astype("Int32")
users_df["d_wk_region_id"] = users_df["d_wk_region_id"].astype("Int32")
users_df["wk_gender"] = users_df["wk_gender"].astype("Int8")
users_df["type"] = users_df["type"].astype("category")
users_df["timezone"] = users_df["timezone"].astype("category")
users_df["subscribed"] = users_df["subscribed"].astype("boolean")
users_df["is_teacher"] = users_df["is_teacher"].astype("boolean")

users_courses_df = pd.read_csv(
    SRC_DIR / "users_courses.csv",
    usecols=[
        "user_id",
        "course_id",
        "state",
        "created_at",
        "updated_at",
        "access_finished_at",
        "wk_points",
        "wk_max_points",
        "wk_max_viewable_lessons",
        "wk_max_task_count",
        "wk_officially_started_at",
        "wk_course_completed_at",
    ],
    parse_dates=[
        "created_at",
        "updated_at",
        "access_finished_at",
        "wk_officially_started_at",
        "wk_course_completed_at",
    ],
    thousands=",",
    low_memory=False,
)
users_courses_df["user_id"] = users_courses_df["user_id"].astype("Int32")
users_courses_df["course_id"] = users_courses_df["course_id"].astype("Int32")
users_courses_df["wk_points"] = users_courses_df["wk_points"].astype("Float32")
users_courses_df["wk_max_points"] = users_courses_df["wk_max_points"].astype("Float32")
users_courses_df["wk_max_viewable_lessons"] = users_courses_df["wk_max_viewable_lessons"].astype("Float32")
users_courses_df["wk_max_task_count"] = users_courses_df["wk_max_task_count"].astype("Float32")
users_courses_df["state"] = users_courses_df["state"].astype("category")

lessons_df = pd.read_csv(
    SRC_DIR / "lessons.csv",
    usecols=[
        "course_id",
        "conspect_expected",
        "task_expected",
        "lesson_number",
        "wk_max_points",
        "wk_task_count",
        "wk_survival_training_expected",
        "wk_scratch_playground_enabled",
        "wk_attendance_tracking_enabled",
        "wk_video_duration",
        "wk_attendance_tracking_disabled_at",
    ],
    parse_dates=["wk_attendance_tracking_disabled_at"],
    thousands=",",
    low_memory=False,
)
lessons_df["course_id"] = lessons_df["course_id"].astype("Int32")
lessons_df["lesson_number"] = lessons_df["lesson_number"].astype("Float32")
lessons_df["wk_max_points"] = lessons_df["wk_max_points"].astype("Float32")
lessons_df["wk_task_count"] = lessons_df["wk_task_count"].astype("Float32")
lessons_df["wk_video_duration"] = lessons_df["wk_video_duration"].astype("Float32")

user_access_histories_df = pd.read_csv(
    SRC_DIR / "user_access_histories.csv",
    usecols=["users_course_id", "access_started_at", "access_expired_at", "activator_class"],
    parse_dates=["access_started_at", "access_expired_at"],
    thousands=",",
    low_memory=False,
)
user_access_histories_df["users_course_id"] = user_access_histories_df["users_course_id"].astype("Int32")
user_access_histories_df["activator_class"] = user_access_histories_df["activator_class"].astype("category")


In [ ]:
# Логи прохождения курса.
user_lessons_df = pd.read_csv(
    SRC_DIR / "user_lessons.csv",
    usecols=[
        "user_id",
        "lesson_id",
        "group_id",
        "video_visited",
        "translation_visited",
        "users_course_id",
        "solved",
        "solved_tasks_count",
        "wk_points",
        "video_viewed",
        "wk_solved_task_count",
    ],
    thousands=",",
    low_memory=False,
)
user_lessons_df["user_id"] = user_lessons_df["user_id"].astype("Int32")
user_lessons_df["lesson_id"] = user_lessons_df["lesson_id"].astype("Int32")
user_lessons_df["group_id"] = user_lessons_df["group_id"].astype("Int32")
user_lessons_df["users_course_id"] = user_lessons_df["users_course_id"].astype("Int32")
user_lessons_df["solved_tasks_count"] = user_lessons_df["solved_tasks_count"].astype("Int16")
user_lessons_df["wk_points"] = user_lessons_df["wk_points"].astype("Float32")
user_lessons_df["wk_solved_task_count"] = user_lessons_df["wk_solved_task_count"].astype("Float32")
user_lessons_df[["video_visited", "translation_visited", "solved", "video_viewed"]] = user_lessons_df[["video_visited", "translation_visited", "solved", "video_viewed"]].astype("boolean")

user_activity_histories_df = pd.read_csv(
    SRC_DIR / "user_activity_histories.csv",
    usecols=["user_lesson_id", "action", "created_at"],
    parse_dates=["created_at"],
    thousands=",",
    low_memory=False,
)
user_activity_histories_df["user_lesson_id"] = user_activity_histories_df["user_lesson_id"].astype("Int32")
user_activity_histories_df["action"] = user_activity_histories_df["action"].astype("category")

wk_users_courses_actions_df = pd.read_csv(
    SRC_DIR / "wk_users_courses_actions.csv",
    usecols=["user_id", "users_course_id", "action", "created_at", "updated_at", "lesson_id"],
    parse_dates=["created_at", "updated_at"],
    thousands=",",
    low_memory=False,
)
wk_users_courses_actions_df["user_id"] = wk_users_courses_actions_df["user_id"].astype("Int32")
wk_users_courses_actions_df["users_course_id"] = wk_users_courses_actions_df["users_course_id"].astype("Int32")
wk_users_courses_actions_df["lesson_id"] = wk_users_courses_actions_df["lesson_id"].astype("Int32")
wk_users_courses_actions_df["action"] = wk_users_courses_actions_df["action"].astype("category")


In [ ]:
# Ответы, тренинги, бейджи и просмотр медиа.
user_answers_df = pd.read_csv(
    SRC_DIR / "user_answers.csv",
    usecols=[
        "user_id",
        "task_id",
        "attempts",
        "solved",
        "points",
        "max_attempts",
        "skipped",
        "resource_type",
        "submitted_at",
        "wk_partial_answer",
        "async_check_status",
    ],
    parse_dates=["submitted_at"],
    thousands=",",
    low_memory=False,
)
user_answers_df["user_id"] = user_answers_df["user_id"].astype("Int32")
user_answers_df["task_id"] = user_answers_df["task_id"].astype("Int32")
user_answers_df["attempts"] = user_answers_df["attempts"].astype("Int8")
user_answers_df["points"] = user_answers_df["points"].astype("Float32")
user_answers_df["max_attempts"] = user_answers_df["max_attempts"].astype("Int8")
user_answers_df["async_check_status"] = user_answers_df["async_check_status"].astype("Int8")
user_answers_df["resource_type"] = user_answers_df["resource_type"].astype("category")
user_answers_df["solved"] = user_answers_df["solved"].map({"True": True, "False": False}).astype("boolean")
user_answers_df["skipped"] = user_answers_df["skipped"].map({"True": True, "False": False}).astype("boolean")
user_answers_df["wk_partial_answer"] = user_answers_df["wk_partial_answer"].map({"True": True, "False": False}).astype("boolean")

user_trainings_df = pd.read_csv(
    SRC_DIR / "user_trainings.csv",
    usecols=[
        "user_id",
        "training_id",
        "solved_tasks_count",
        "earned_points",
        "type",
        "state",
        "submitted_answers_count",
        "started_at",
        "finished_at",
        "attempts",
        "mark",
        "mark_saved_at",
    ],
    parse_dates=["started_at", "finished_at", "mark_saved_at"],
    thousands=",",
    low_memory=False,
)
user_trainings_df["user_id"] = user_trainings_df["user_id"].astype("Int32")
user_trainings_df["training_id"] = user_trainings_df["training_id"].astype("Int32")
user_trainings_df["solved_tasks_count"] = user_trainings_df["solved_tasks_count"].astype("Int16")
user_trainings_df["earned_points"] = user_trainings_df["earned_points"].astype("Float32")
user_trainings_df["submitted_answers_count"] = user_trainings_df["submitted_answers_count"].astype("Int16")
user_trainings_df["attempts"] = user_trainings_df["attempts"].astype("Int8")
user_trainings_df["mark"] = user_trainings_df["mark"].astype("Float32")
user_trainings_df["type"] = user_trainings_df["type"].astype("category")
user_trainings_df["state"] = user_trainings_df["state"].astype("category")

user_award_badges_df = pd.read_csv(
    SRC_DIR / "user_award_badges.csv",
    usecols=["award_badge_id", "user_id", "created_at"],
    parse_dates=["created_at"],
    thousands=",",
    low_memory=False,
)
user_award_badges_df["award_badge_id"] = user_award_badges_df["award_badge_id"].astype("Int8")
user_award_badges_df["user_id"] = user_award_badges_df["user_id"].astype("Int32")

award_badges_df = pd.read_csv(
    SRC_DIR / "award_badges.csv",
    usecols=["title", "level", "quota", "special"],
    low_memory=False,
)
award_badges_df["title"] = award_badges_df["title"].astype("category")
award_badges_df["level"] = award_badges_df["level"].astype("Int8")
award_badges_df["quota"] = award_badges_df["quota"].astype("Int16")
award_badges_df["special"] = award_badges_df["special"].astype("boolean")

wk_media_view_sessions_df = pd.read_csv(
    SRC_DIR / "wk_media_view_sessions.csv",
    usecols=["resource_type", "viewer_id", "segments_total", "viewed_segments_count", "started_at", "kind"],
    parse_dates=["started_at"],
    thousands=",",
    low_memory=False,
)
wk_media_view_sessions_df["resource_type"] = wk_media_view_sessions_df["resource_type"].astype("category")
wk_media_view_sessions_df["viewer_id"] = wk_media_view_sessions_df["viewer_id"].astype("Int32")
wk_media_view_sessions_df["segments_total"] = wk_media_view_sessions_df["segments_total"].astype("Int16")
wk_media_view_sessions_df["viewed_segments_count"] = wk_media_view_sessions_df["viewed_segments_count"].astype("Int16")
wk_media_view_sessions_df["kind"] = wk_media_view_sessions_df["kind"].astype("category")


In [ ]:
dataframes = {
    "users_df": users_df,
    "users_courses_df": users_courses_df,
    "lessons_df": lessons_df,
    "user_access_histories_df": user_access_histories_df,
    "user_lessons_df": user_lessons_df,
    "user_activity_histories_df": user_activity_histories_df,
    "user_answers_df": user_answers_df,
    "user_trainings_df": user_trainings_df,
    "wk_users_courses_actions_df": wk_users_courses_actions_df,
    "wk_media_view_sessions_df": wk_media_view_sessions_df,
    "award_badges_df": award_badges_df,
    "user_award_badges_df": user_award_badges_df,
}

import_summary_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "rows": [df.shape[0] for df in dataframes.values()],
        "cols": [df.shape[1] for df in dataframes.values()],
        "memory_mb": [round(df.memory_usage(deep=True).sum() / 1024**2, 2) for df in dataframes.values()],
    }
).sort_values("memory_mb", ascending=False, ignore_index=True)

total_memory_mb = round(import_summary_df["memory_mb"].sum(), 2)

display(import_summary_df)
display(f"Суммарно в памяти: {total_memory_mb} MB")
